# Sample RAG Pipeline — Granite Switch Adapters

> *Corpus:* IBM mt-rag-benchmark government-services passages (49k docs).

**Duration:** ~30 min (first run, includes corpus embedding)

This notebook demonstrates a **conversational RAG pipeline** where every AI capability — guardian checks, query rewriting, retrieval-grounded answering, citations — runs through a single vLLM endpoint. The intrinsics are embedded adapters inside the Granite Switch model, activated by control tokens at inference time.

*Why vLLM:* much faster inference in production environments; HF support for Granite Switch in mellea coming.

**What you'll learn:**
- How to build a 7-turn conversation that exercises every step of the pipeline - grounded answers with citations, clarification on ambiguous queries, early exit on unanswerable ones, and guardian blocks for out-of-scope or harmful requests.
- How to chain multiple intrinsics (guardian, query rewrite, answerability, clarification, grounded generation, citations) into one RAG pipeline.
- How control tokens route each intrinsic call to the right embedded adapter without loading separate models.
- How to handle the four terminal states — `blocked`, `unanswerable`, `needs_clarification`, and `done` — in a stateful conversation.
- How to lift `run_pipeline` out of this notebook and drop it into your own app.

---
**Prerequisites:** GPU runtime (T4 or better). Go to *Runtime → Change runtime type → T4 GPU*.

New to mellea intrinsics? Start with [`hello_mellea.ipynb`](./hello_mellea.ipynb) for a softer walkthrough of each intrinsic in isolation. Full setup details (GPU sizes, HF auth, multi-GPU) are in [`PREREQUISITES.md`](../PREREQUISITES.md).

## 0 · Install and set up

In [ ]:
# Install granite-switch with tutorial dependencies (includes vLLM backend).
%pip install -q "granite-switch[tutorials]"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # needed to pull ibm-granite models from the Hub

In [ ]:
from granite_switch.tutorials.vllm_server import kill_stale_vllm_processes, launch_vllm, print_gpu_state, tail_log, wait_for_server

kill_stale_vllm_processes()
print_gpu_state()

## 1 · Launch vLLM server

Start the Granite Switch model on port 8000. The server runs in the background; `wait_for_server` polls `/health` until it's ready.

In [ ]:
VLLM_MODEL = "ibm-granite/granite-switch-4.1-3b-preview"
VLLM_PORT = 8000
MAX_MODEL_LEN = 30720  # 30k

vllm_proc = launch_vllm(
    model=VLLM_MODEL,
    port=VLLM_PORT,
    max_model_len=MAX_MODEL_LEN,
    log_file="/content/vllm_server.log",
)
if not wait_for_server(VLLM_PORT):
    tail_log("/content/vllm_server.log")

**Intrinsics used in this pipeline:** each row is one embedded adapter, invoked via mellea.

| Intrinsic | Role in the pipeline |
|-----------|----------------------|
| `guardian_check` (harm)       | Classifies the full conversation for harmful content; blocks before any retrieval. |
| `guardian_check` (scope)      | Classifies whether the query is about government services; blocks out-of-scope topics. |
| `rag.rewrite_question`        | Rewrites the current turn into a standalone query using conversation history. |
| `rag.check_answerability`     | Decides whether the retrieved documents can actually answer the query. |
| `rag.clarify_query`           | Returns `CLEAR` if the query is unambiguous, otherwise a follow-up question. |
| `mfuncs.act` (base model)     | Generates the final grounded answer from the retrieved documents. |
| `rag.find_citations`          | Maps spans of the answer back to supporting passages in the retrieved docs. |

**Pipeline steps:** the diagram below traces a single user turn. Diamonds are the decision gates (harm, scope, answerability, clarification); rounded nodes are the four terminal states.

In [ ]:
mermaid_diagram = """
flowchart TD
    Q([User query]) --> G1{"[1a] Guardian — Harm<br/>is the query harmful?"}
    G1 -- "score ≥ 0.5" --> BLOCKED([⛔ BLOCKED])
    G1 -- safe --> G2{"[1b] Guardian — Scope<br/>is it about government services?"}
    G2 -- "score < 0.5" --> BLOCKED
    G2 -- in-scope --> REWRITE["[2] Query Rewrite<br/>disambiguate using history"]
    REWRITE --> RETRIEVE["[3] ChromaDB Retrieval<br/>top-K dense search"]
    RETRIEVE --> ANS{"[4] Answerability<br/>can docs answer?"}
    ANS -- unanswerable --> UNANS([🔍 UNANSWERABLE])
    ANS -- answerable --> CLAR{"[5] Clarification<br/>is the query clear enough?"}
    CLAR -- unclear --> NEEDCLAR([❓ NEEDS CLARIFICATION])
    CLAR -- CLEAR --> GEN["[6] Answer<br/>grounded generation"]
    GEN --> CITE["[7] Citations<br/>supporting spans"]
    CITE --> DONE([✅ DONE])
"""

import base64
from IPython.display import Image, display

def mermaid(graph: str):
    graphbytes = graph.encode("utf8")
    base64_bytes = base64.urlsafe_b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    display(Image(url="https://mermaid.ink/img/" + base64_string, width=400))

mermaid(mermaid_diagram)

## 2 · Configuration
Endpoints, model IDs, and corpus paths. Every value falls back to a sensible default, so the cell runs as-is if your vLLM server is on `localhost:8000`.

In [ ]:
import json
import logging
import os
import warnings
from functools import partial
from pathlib import Path

from IPython.display import display, Markdown
from granite_switch.tutorials.govt_data_loader import load_or_build_govt_chroma
from granite_switch.tutorials.rag_display import show_answer, show_history, _is_clear
from granite_switch.tutorials.rag_display import show_intermediates_sequential
from mellea.backends import ModelOption
from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.components import Document as MelleaDocument
from mellea.stdlib.components.chat import Message as MelleaMessage
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.components.intrinsic.guardian import guardian_check
from mellea.stdlib.context import ChatContext
import mellea.stdlib.functional as mfuncs

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"), override=False)
except ImportError:
    pass

# ── vLLM server ───────────────────────────────────────────────────────────────
# URL of the running vLLM OpenAI-compatible endpoint.
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")

# Model name as reported by GET /v1/models (usually the path/repo used at launch).
VLLM_MODEL_NAME = os.environ.get("VLLM_MODEL_NAME", "ibm-granite/granite-switch-4.1-3b-preview")

# HF Hub repo ID (or local path) to load I/O configs for the embedded adapters.
GRANITE_SWITCH_SOURCE = os.environ.get("GRANITE_SWITCH_SOURCE", VLLM_MODEL_NAME)

# Guardian: which safety criterion to evaluate
GUARDIAN_CRITERIA = "harm"  # harm | social_bias | groundedness | jailbreak | ...

# ── Embedding model (used to build + query ChromaDB) ─────────────────────────
EMBEDDING_MODEL_ID = "ibm-granite/granite-embedding-small-english-r2"

# ── ChromaDB persistence path ─────────────────────────────────────────────────
# Share this directory (zipped) to skip the extraction step entirely.
CHROMA_PATH = "./govt_chroma"

# ── Corpus source (only needed when building the index from scratch) ─────────
# govt.jsonl: 49k government-service passages from IBM mt-rag-benchmark.
GOVT_JSONL_URL  = "https://github.com/IBM/mt-rag-benchmark/raw/main/corpora/passage_level/govt.jsonl.zip"
GOVT_JSONL_PATH = "./govt.jsonl"

# ── Retrieval ─────────────────────────────────────────────────────────────────
# TOP_K balances recall (more candidates -> better chance of a relevant passage)
# against context budget (every doc gets passed through answerability, clarification,
# generation, and citation prompts). 20 is the mt-rag-benchmark default.
TOP_K = 10

# Bind TOP_K so query cells can call `show_intermediates(r)` without repeating it.
show_intermediates = partial(show_intermediates_sequential, top_k=TOP_K)

print(f"vLLM:      {VLLM_BASE_URL}  ({VLLM_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_MODEL_ID}")
print(f"ChromaDB:  {CHROMA_PATH}")

## 3 · Build or load vector corpus
Data prep is delegated to `scripts/utils/govt_data_loader.py` to keep this notebook focused on the RAG pipeline.

**First run:** downloads ~50 MB and embeds the corpus passages. **Subsequent runs:** load the persisted index instantly.

> **Note:** to keep the tutorial fast, we filter most non-related docs and embed only the curated subset that the demo queries actually retrieve. For a full corpus load, set `load_only_tutorial_docs=False` in the call below.

In [ ]:
# Load or build the ChromaDB corpus.
# First run: downloads govt.jsonl.zip from IBM mt-rag-benchmark (49k passages),
# embeds with `ibm-granite/granite-embedding-small-english-r2` into ./govt_chroma.
# Subsequent runs: loads ./govt_chroma instantly.
#
# `load_only_tutorial_docs=True` restricts embedding to the curated subset
# the demo queries actually retrieve. Set False to embed the full corpus.

chroma_collection = load_or_build_govt_chroma(
    chroma_path             = CHROMA_PATH,
    jsonl_path              = GOVT_JSONL_PATH,
    jsonl_url               = GOVT_JSONL_URL,
    embedding_model_id      = EMBEDDING_MODEL_ID,
    device                  = "cpu",
    load_only_tutorial_docs = True,
)


## 4 · Connect to vLLM backend
Registers the Granite Switch embedded adapters from `GRANITE_SWITCH_SOURCE`
so all intrinsics (guardian, RAG, citations) route through the correct control tokens.

In [ ]:
backend = OpenAIBackend(
    model_id=VLLM_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="unused",
)
backend.register_embedded_adapter_model(GRANITE_SWITCH_SOURCE)
print(f"Backend ready - {backend.list_adapters()}")

## 5 · The pipeline function
`run_pipeline(query, ctx)` is the whole pipeline - guardian, rewrite, retrieve, answerability, clarify, answer, citations - with one exit per terminal state. Sub-cell 6a quiets mellea's INFO/WARNING logs so the pipeline output is readable; the display helpers themselves were imported in section 3.

In [ ]:
# ── Guardian criteria ─────────────────────────────────────────────────────────────────────────────────
# Scope check: positive definition - message should be about government services.
GUARDIAN_SCOPE_CRITERIA = (
    "Governmental services content refers to messages concerning services "
    "that are provided, administered, funded, or regulated by a government "
    "agency at any level - federal, state, local, or municipal. This "
    "includes taxes and tax filings, public benefits (such as social "
    "security, disability benefits, unemployment, food assistance, Medicaid), "
    "permits and licenses, voting and elections, immigration, public healthcare "
    "programs, housing assistance, veterans affairs, public records, "
    "court and legal processes, and direct interactions with any "
    "government office or program."
)

# Late instruction appended to the user message at generation time.
# Placed after history + documents to preserve KV cache prefix sharing.
GENERATION_INSTRUCTION = (
    "Answer concisely and directly based only on the provided documents. "
    "Do not repeat the question or add unnecessary preamble."
)


# ── Full pipeline ───────────────────────────────────────────────────────────────────────────────
def run_pipeline(query, ctx):
    """Run one turn of the RAG pipeline.

    Prints the answer, appends the turn to `ctx` (unless blocked), and
    returns `(ctx, r)`.
    """
    n_msgs = len(ctx.as_list())
    print(f"[history before: {n_msgs} msg(s)]")

    r = {"query": query, "blocked": False, "unanswerable": False, "needs_clarification": False}

    ctx_with_query = ctx.add(MelleaMessage("user", query))

    # ctx_with_query (history + current msg) feeds steps that classify or ground
    # against the full turn (guardian, citations); the rest take ctx (history only)
    # plus the query as an explicit argument, so it doesn't need to live in ctx.

    # [1a] Harm check - must run *before* the scope check so a query that is both
    # harmful and out-of-scope (e.g. "how do I forge a government ID?") is labeled
    # harmful rather than merely out-of-scope. Harm criterion name comes from the
    # mellea guardian dictionary.
    r["guardian_harm_score"] = guardian_check(ctx_with_query, backend, GUARDIAN_CRITERIA, target_role="user")
    if r["guardian_harm_score"] >= 0.5:
        r["blocked"] = True
        r["block_reason"] = f"Harmful content detected (score={r['guardian_harm_score']:.3f})"
        show_answer(r)
        return ctx, r

    # [1b] Scope check - is the query about government services? Criteria defined above in GUARDIAN_SCOPE_CRITERIA.
    r["guardian_scope_score"] = guardian_check(ctx_with_query, backend, GUARDIAN_SCOPE_CRITERIA, target_role="user")
    if r["guardian_scope_score"] < 0.5:
        r["blocked"] = True
        r["block_reason"] = f"Out of scope - not a government services topic (score={r['guardian_scope_score']:.3f})"
        show_answer(r)
        return ctx, r

    # [2] Rewrite query using conversation history.
    r["rewritten_query"] = rag.rewrite_question(query, ctx, backend)

    # [3] Retrieve candidate documents from ChromaDB.
    r["documents"] = chroma_collection.query(query_texts=[r["rewritten_query"]], n_results=TOP_K)["documents"][0]
    mellea_docs = [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(r["documents"])]

    # [4] Answerability - can the retrieved docs answer the query?
    r["answerability"] = rag.check_answerability(r["rewritten_query"], mellea_docs, ctx, backend)
    if r["answerability"] == "unanswerable":
        r["unanswerable"] = True
    else:
        # [5] Clarification - if the query is still ambiguous, ask a follow-up question instead of guessing.
        r["clarification"] = rag.clarify_query(r["rewritten_query"], mellea_docs, ctx, backend)
        if not _is_clear(r["clarification"]):
            r["needs_clarification"] = True
        else:
            # [6] Answer - generate grounded response from retrieved documents.
            prompted_query = r["rewritten_query"] + "\n\n" + GENERATION_INSTRUCTION
            out, _ = mfuncs.act(
                MelleaMessage("user", prompted_query, documents=mellea_docs),
                ctx, backend,
                # Deterministic decoding: for grounded RAG we want the model to repeat what's
                # in the docs, not paraphrase creatively. This also makes demos reproducible.
                model_options={ModelOption.TEMPERATURE: 0.0},
            )
            r["answer"] = str(out)

            # [7] Citations - map answer spans to supporting document passages.
            r["citations"] = rag.find_citations(r["answer"], mellea_docs, ctx_with_query, backend) if mellea_docs else []

    show_answer(r)

    # Append this turn to history. (Blocked turns returned earlier and are not recorded.)
    if r["unanswerable"]:
        reply = "I don't have enough information in my knowledge base to answer that."
    elif r["needs_clarification"]:
        reply = r["clarification"]
    else:
        reply = r.get("answer", "")
    user_docs = [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(r["documents"])]
    ctx = ctx.add(MelleaMessage("user", query, documents=user_docs))
    ctx = ctx.add(MelleaMessage("assistant", reply))
    print(f"-> history now has {len(ctx.as_list())} message(s)")
    return ctx, r


print("run_pipeline ready.")

### 5a · Display helpers (printing only - not part of the pipeline)
These functions format and print pipeline results as Markdown. **You can skip reading this cell** - it contains no pipeline logic, only display utilities (`show_answer`, `show_intermediates`, `show_history`).

In [ ]:
# Display helpers (show_answer, show_intermediates, show_history, Conversation) live
# in tutorials/scripts/utils/rag_display.py — already imported in section 1.

# Silence mellea's INFO/WARNING chatter (TemplateRepresentation, "Tools for call", SUCCESS)
logging.getLogger("mellea").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*TemplateRepresentation.*")

print("Display helpers loaded from rag_display; mellea logging quieted.")

## 6 · Queries
Each cell is one turn. History accumulates automatically.
- `run_pipeline(query, ctx)` - run pipeline, show the final answer, update history, return `(ctx, r)`.
- `show_intermediates(r)` - step-by-step breakdown for any result.
- `show_history(conv)` - print the full conversation so far.

<details open><summary>📖 Reference: what <code>show_intermediates(r)</code> displays at each step</summary>

Each row describes what `show_intermediates(r)` renders for one step of the pipeline. In the demo cells below, `r1` through `r6` hold the result of each turn.

| Step | What you'll see |
|------|-----------------|
| **[1a] Guardian - Harm**  | `🟢 safe` / `🔴 harmful` + raw score. Exits early (blocks) if score >= 0.5. |
| **[1b] Guardian - Scope** | `🟢 in-scope` / `🔴 out-of-scope` + raw score. Exits early (blocks) if score < 0.5. |
| **[2] Query Rewrite** | Original query vs. the rewritten version (disambiguated using conversation history). |
| **[3] ChromaDB Retrieval** | Number of documents retrieved; each document is collapsible. |
| **[4] Answerability** | `✅ answerable` / `🔍 unanswerable` + verdict string. Exits early if unanswerable. |
| **[5] Clarification** | `✅ CLEAR` / `❓ needs clarification` + the follow-up question the model would ask. Exits early if not CLEAR. |
| **[6] Answer** | Full model response + character count. Only appears when the pipeline reaches the end. |
| **[7] Citations** | JSON list of document spans that support the answer. Shows *(none)* if the model didn't attribute any. |

</details>

In [ ]:
# Q1 - clarification: "the government service" is ambiguous - it could mean the IRS,
# the California FTB, or another agency. Retrieval pulls docs from BOTH the IRS and
# the FTB; the clarification adapter recognizes the multi-referent ambiguity and asks
# which one. The rewriter is correctly a no-op (rewriting away the ambiguity would
# defeat the clarification step).
ctx = ChatContext()
ctx, r1 = run_pipeline("How long does it take for the government service to refund?", ctx)
show_intermediates(r1)

In [ ]:
# Q2 - resolves the clarification: a 2-token reply ("The IRS") is enough for the
# rewriter to expand into a full standalone query using Q1 history, which then
# retrieves IRS-specific docs and produces a grounded answer.
ctx, r2 = run_pipeline("The IRS", ctx)
show_intermediates(r2)

In [ ]:
# Q3 - history-aware rewrite: "paper return instead" only makes sense relative to
# the IRS-refund thread established in Q1-Q2. The rewrite adapter uses ctx to
# produce a standalone query about paper returns, which is what gets sent to retrieval.
ctx, r3 = run_pipeline("What if I'm filing a paper return instead?", ctx)
show_intermediates(r3)

In [ ]:
# Q4 - cross-turn pronoun resolution: "amending it" - the rewriter resolves "it"
# to the paper return from Q3's rewritten query, demonstrating that ctx flows
# through multiple turns of history.
ctx, r4 = run_pipeline("And what's the deadline for amending it?", ctx)
show_intermediates(r4)

In [ ]:
# Q5 - rewrite + unanswerable: "it" refers back to the amended paper return from Q4.
# Rewriter resolves the pronoun ("How much does it cost to amend a paper tax return?")
# but the corpus has no fee schedule, so answerability routes to unanswerable instead
# of hallucinating a number.
ctx, r5 = run_pipeline("How much does it cost?", ctx)
show_intermediates(r5)

In [ ]:
# Q6 - out of scope: weather is not a government services topic; guardian blocks it.
ctx, r6 = run_pipeline("What's the weather in New York tomorrow?", ctx)
show_intermediates(r6)

In [ ]:
# Q7 - harm: guardian blocks request to forge government documents.
ctx, r7 = run_pipeline("How do I forge a government ID?", ctx)
show_intermediates(r7)

In [ ]:
show_history(ctx)

## 7 · Next steps

- **Compose your own checkpoint.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) walks through building a model from the IBM adapter libraries.
- **Watch ALORA vs LoRA race.** [`alora_vs_lora_race.ipynb`](./alora_vs_lora_race.ipynb) compares the two activation styles head-to-head on the same workload.